In [1]:
import requests
from requests.structures import CaseInsensitiveDict
import numpy_financial as npf
import numpy as np
from datetime import datetime as dt
import pandas as pd

headers = CaseInsensitiveDict()
headers["accept"] = "application/json"
baseUrlApi = "https://fintual.cl/api/real_assets/"

class Portfolio:

    def __init__(self, id, doc = True):
        urlAssetInfo         = f"{baseUrlApi}{id}"
        assetInfo            = requests.get(urlAssetInfo,  headers=headers).json()['data']['attributes']

        self.id          = id
        self.name        = assetInfo['name']
        self.startDate   = assetInfo['start_date'] 
        self.lastDate    = pd.to_datetime(assetInfo['last_day']['date'], format='%Y-%m-%d')
        # self.lastDate    = assetInfo['last_day']['date'] 
        self.lastPrice   = assetInfo['last_day']["net_asset_value"]
        # self.df          = pd.DataFrame() 
        # self.df          = self.get_all_days()

        # Url de la api de funtual necesario para obtener todos los valores existentes en el tiempo
        
    
       
        
        # def get_api_info(self, date=""):
            ### Existe El documento
                ### Si existe documento se valida si los datos están actualizados
                    ### Si estan actualizados se finaliza el proceso
                    ### Si los datos NO estan actualizados se buscan la diferencia de datos y se le acopla a al dataframe
                ### En caso de No existir el documento se buscar todos los datos del protafolio y se crea
            ### Verificar si existe el nombre del portafolio en el documento



        if doc == False: # True
            # Revisar y actualizar datos
            days = f"?from_date={2024-12-22}"
            pass
        else:
            # Crear archivo
            days = f""
            pass


        urlAssetInfoDays  = f"{baseUrlApi}{id}/days{days}?to_date=2024-08-28" #?to_date=2024-08-28
        # urlAssetInfoDays  = f"{baseUrlApi}{id}/days" #?to_date=2024-08-28
        # https://fintual.cl/api/real_assets/186/days?from_date=2024-12-20

        # obtenemos los datos de la api y lo pasamos a un DataFrame
        assetInfoDays        = requests.get(urlAssetInfoDays,  headers=headers).json()['data']
        df = pd.DataFrame(assetInfoDays)

        # Nomalizamos los datos a Json
        df = pd.json_normalize(df['attributes'])[['date', 'price', 'shareholders', 'total_assets', 'total_net_assets', 'outstanding_shares']]

        # Elimina las filas las cuales no dispongan del datos clave 'total_assets'
        df.dropna(subset=['total_assets'], inplace=True)

        # Se cambia el formato de la columna fecha para generar filtros y crear nuevas columnas con los años, mese y día
        df['date']  = pd.to_datetime(df['date'],format='%Y-%m-%d')
        df['year']  = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['day']   = df['date'].dt.day
        
        # Cambio de nombre de las columas 
        df.columns = ('fecha','precio','accionistas','activos_totales','activos_neto_totales','acciones_en_circulación', 'año', 'mes', 'día')

        df = df[::-1].reset_index(drop=True)

        self.df = df 
        self.info = pd.DataFrame({
            'Detalle' : [self.name, self.id, self.df.iloc[-1]['fecha'] ]
        }).T
        self.info.columns = ['name', 'id', 'last_date']                 



        
    def __str__(self):
        return self.name

In [ ]:
import pandas as pd

archivo_excel = "portafolio.xlsx"
id_portfolio = [186,187,188, 15077]

try:

    ######### Se adquiere los datos del archivo
    di_port = {}
    excel_file = pd.ExcelFile(archivo_excel)
    hojas = excel_file.sheet_names 
    for hoja in hojas:
        df = excel_file.parse(hoja)  # Leer cada hoja como un DataFrame
        datos_tabla = df.head(1).iloc[:,:3] 
        di_port = {datos_tabla['id']: }
       



        ######### se compara los id ingresado junto a los id que se encuentan en el documento


            ######### Si existe en el documento se manda la última fecha para que los datos sean actualizados
        resultado = df[df.iloc[:,0]== 'fecha'].index[0]
        resultado = df.iloc[resultado:]

            ######### En caso de no existir el id en el documento, se crea desde 0





    # # # La información se actualizará 
    # # excel_file = pd.ExcelFile(archivo_excel)
    # # hojas = excel_file.sheet_names  
    # # # info = pd.read_excel(archivo_excel)
    # print(hojas)
    # # info = pd.read_excel(f"precios.xlsx",sheet_name='Risky Norris')

    # # Recorrer las hojas y obtener su contenido
    # for hoja in hojas:
    #     # print(f"Contenido de la hoja: {hoja}")
    #     df = excel_file.parse(hoja)  # Leer cada hoja como un DataFrame
    #     datos_tabla = df.head(1).iloc[:,:3]
        
    #     # print(datos_tabla)  # Muestra las primeras filas
    #     # print(datos_tabla['id'].isin(id_portfolio) )  # Muestra las primeras filas
    #     print()
        
    # # # info = pd.read_excel(f"{archivo_excel}",sheet_name='Risky Norris')
    # # print(f"El archivo '{archivo_excel}' existe y es un archivo Excel válido.")

except FileNotFoundError:
    # Se creará el documento
    print(f"El archivo '{archivo_excel}' no existe.")
    li_port = []
    for n in id_portfolio:
        li_port.append(Portfolio(n))

    with pd.ExcelWriter(archivo_excel) as writer:
        for port_n in li_port:
            port_n.info.to_excel(writer, sheet_name=f"{port_n.name}", index=False)
            startrow = 3
            port_n.df.to_excel(writer, sheet_name=f"{port_n.name}", startrow=startrow, index=False) 

except Exception as e:
    #Se mostrará el error
    print(f"Ocurrió un error al intentar leer el archivo: {e}")

--                     name         id    last_date       Unnamed: 3  \
2                   fecha     precio  accionistas  activos_totales   
3     2018-02-13 00:00:00  1003.8325            1           708357   
4     2018-02-14 00:00:00  1013.2619            2          1291954   
5     2018-02-15 00:00:00  1016.1704            2          1133173   
6     2018-02-16 00:00:00  1016.2871            2          1133340   
...                   ...        ...          ...              ...   
2387  2024-08-24 00:00:00  2661.8805        53590     325170342960   
2388  2024-08-25 00:00:00  2661.7973        53590     325170749998   
2389  2024-08-26 00:00:00   2631.319        53606     324695617880   
2390  2024-08-27 00:00:00  2636.8129        53591     322465959285   
2391  2024-08-28 00:00:00  2625.0202        53604     320818066813   

                Unnamed: 4               Unnamed: 5 Unnamed: 6 Unnamed: 7  \
2     activos_neto_totales  acciones_en_circulación        año        mes   
3  

In [3]:
port186 = Portfolio(186)
# port186.get_all_days()
dt186 = port186.df

In [4]:
dt186

,fecha,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación,año,mes,día
0,2018-02-13,1003.8325,1.0,7.083570e+05,4.015330e+05,4.000000e+02,2018,2,13
1,2018-02-14,1013.2619,2.0,1.291954e+06,9.853050e+05,9.724088e+02,2018,2,14
2,2018-02-15,1016.1704,2.0,1.133173e+06,1.133133e+06,1.115101e+03,2018,2,15
3,2018-02-16,1016.2871,2.0,1.133340e+06,1.133263e+06,1.115101e+03,2018,2,16
4,2018-02-17,1016.2728,2.0,1.133361e+06,1.133247e+06,1.115101e+03,2018,2,17
...,...,...,...,...,...,...,...,...,...
2384,2024-08-24,2661.8805,53590.0,3.251703e+11,2.392019e+11,8.986201e+07,2024,8,24
2385,2024-08-25,2661.7973,53590.0,3.251707e+11,2.391945e+11,8.986201e+07,2024,8,25
2386,2024-08-26,2631.3190,53606.0,3.246956e+11,2.364713e+11,8.986797e+07,2024,8,26
2387,2024-08-27,2636.8129,53591.0,3.224660e+11,2.368288e+11,8.981631e+07,2024,8,27


In [5]:
print(dt186.iloc[-1]['fecha']) 

2024-08-28 00:00:00


In [6]:
print(dt186.iloc[-1]['fecha'])        
print(port186.lastDate)

print(type(dt186.iloc[-1]['fecha']))  
print(type(port186.lastDate))

dt186.iloc[-1]['fecha'] <= port186.lastDate

2024-08-28 00:00:00
2024-12-26 00:00:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
<class 'pandas._libs.tslibs.timestamps.Timestamp'>


True

In [7]:
dt186[dt186['fecha'] < dt186.iloc[-1]['fecha']]
# dt186[dt186['fecha'] < port186.lastDate]

,fecha,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación,año,mes,día
0,2018-02-13,1003.8325,1.0,7.083570e+05,4.015330e+05,4.000000e+02,2018,2,13
1,2018-02-14,1013.2619,2.0,1.291954e+06,9.853050e+05,9.724088e+02,2018,2,14
2,2018-02-15,1016.1704,2.0,1.133173e+06,1.133133e+06,1.115101e+03,2018,2,15
3,2018-02-16,1016.2871,2.0,1.133340e+06,1.133263e+06,1.115101e+03,2018,2,16
4,2018-02-17,1016.2728,2.0,1.133361e+06,1.133247e+06,1.115101e+03,2018,2,17
...,...,...,...,...,...,...,...,...,...
2383,2024-08-23,2661.9637,53590.0,3.251699e+11,2.392094e+11,8.986201e+07,2024,8,23
2384,2024-08-24,2661.8805,53590.0,3.251703e+11,2.392019e+11,8.986201e+07,2024,8,24
2385,2024-08-25,2661.7973,53590.0,3.251707e+11,2.391945e+11,8.986201e+07,2024,8,25
2386,2024-08-26,2631.3190,53606.0,3.246956e+11,2.364713e+11,8.986797e+07,2024,8,26


In [8]:
baseUrlApi = "https://fintual.cl/api/real_assets/"
id=186
url_info  = f"{baseUrlApi}{id}/days?from_date={str(dt186.iloc[-1]['fecha'])[:10]}" #?to_date=2024-08-28
        # url_info  = f"{baseUrlApi}{id}/days" #?to_date=2024-08-28
        # https://fintual.cl/api/real_assets/186/days?from_date=2024-12-20

        # obtenemos los datos de la api y lo pasamos a un DataFrame
info_day_1        = requests.get(url_info,  headers=headers).json()['data']
df2 = pd.DataFrame(info_day_1)
df2

,id,type,attributes
0,186-2024-08-28,real_asset_day,"{'date': '2024-08-28', 'price': 2625.0202, 'fi..."
1,186-2024-08-29,real_asset_day,"{'date': '2024-08-29', 'price': 2641.3884, 'fi..."
2,186-2024-08-30,real_asset_day,"{'date': '2024-08-30', 'price': 2652.1236, 'fi..."
3,186-2024-08-31,real_asset_day,"{'date': '2024-08-31', 'price': 2652.0396, 'fi..."
4,186-2024-09-01,real_asset_day,"{'date': '2024-09-01', 'price': 2651.9557, 'fi..."
...,...,...,...
116,186-2024-12-22,real_asset_day,"{'date': '2024-12-22', 'price': 3029.92, 'fixe..."
117,186-2024-12-23,real_asset_day,"{'date': '2024-12-23', 'price': 3060.0264, 'fi..."
118,186-2024-12-24,real_asset_day,"{'date': '2024-12-24', 'price': 3083.318, 'fix..."
119,186-2024-12-25,real_asset_day,"{'date': '2024-12-25', 'price': 3083.2207, 'fi..."


In [9]:
urlAssetInfoDays2 = f"{baseUrlApi}{id}/days?to_date=2024-12-22" #?to_date=2024-08-28
assetInfoDays2        = requests.get(urlAssetInfoDays2,  headers=headers).json()['data']
ds = pd.DataFrame(assetInfoDays2)
ds

,id,type,attributes
0,186-2024-12-22,real_asset_day,"{'date': '2024-12-22', 'price': 3029.92, 'fixe..."
1,186-2024-12-21,real_asset_day,"{'date': '2024-12-21', 'price': 3030.0167, 'fi..."
2,186-2024-12-20,real_asset_day,"{'date': '2024-12-20', 'price': 3030.1134, 'fi..."
3,186-2024-12-19,real_asset_day,"{'date': '2024-12-19', 'price': 3013.5461, 'fi..."
4,186-2024-12-18,real_asset_day,"{'date': '2024-12-18', 'price': 2997.9504, 'fi..."
...,...,...,...
2501,186-2018-02-16,real_asset_day,"{'date': '2018-02-16', 'price': 1016.2871, 'fi..."
2502,186-2018-02-15,real_asset_day,"{'date': '2018-02-15', 'price': 1016.1704, 'fi..."
2503,186-2018-02-14,real_asset_day,"{'date': '2018-02-14', 'price': 1013.2619, 'fi..."
2504,186-2018-02-13,real_asset_day,"{'date': '2018-02-13', 'price': 1003.8325, 'fi..."


In [10]:
info = pd.read_excel(f"precios.xlsx",sheet_name='Risky Norris')
pd.ExcelFile()

FileNotFoundError: [Errno 2] No such file or directory: 'precios.xlsx'